# License
This notebook is licensed under the MIT License. See the LICENSE file in the repository root for details.

In [ ]:
# Setup for Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q networkx matplotlib pandas Faker
    print('Running in Colab')

# Network Centrality Measures In-Depth

Centrality measures allow us to identify the most important nodes in a network. In social networks, these might be influencers, bridges between communities, or efficient communicators.

In this notebook, we will:
1.  Generate a **Synthetic Social Network** (Scale-Free) using the Barabási-Albert model.
2.  Assign fictitious identities to nodes using `Faker`.
3.  Calculate key centrality measures:
    *   **Degree Centrality**: Popularity (number of connections).
    *   **Betweenness Centrality**: Control of information flow (bridges).
    *   **Closeness Centrality**: Efficiency (average distance to others).
    *   **Eigenvector Centrality**: Influence (connected to other well-connected nodes).
4.  Visualize the network highlighting influential nodes.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from faker import Faker

fake = Faker()
Faker.seed(42)
np.random.seed(42)

## 1. Network Generation

Real-world social networks often follow a **Power Law** distribution (Scale-Free), where a few nodes (hubs) have many connections. We simulate this using the **Barabási-Albert** model.

In [ ]:
n_nodes = 50
m_edges = 2  # Number of edges to attach from a new node to existing nodes

G = nx.barabasi_albert_graph(n_nodes, m_edges, seed=42)

# Assign Names
node_names = {i: fake.first_name() for i in G.nodes()}
nx.set_node_attributes(G, node_names, 'name')

print(f"Generated Network with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

## 2. Calculating Centrality

We compute multiple metrics to understand different facets of "importance".

In [ ]:
# Degree
degree = nx.degree_centrality(G)

# Betweenness
betweenness = nx.betweenness_centrality(G)

# Closeness
closeness = nx.closeness_centrality(G)

# Eigenvector
eigenvector = nx.eigenvector_centrality(G)

# Compile into DataFrame
data = []
for node_id in G.nodes():
    data.append({
        'ID': node_id,
        'Name': G.nodes[node_id]['name'],
        'Degree': degree[node_id],
        'Betweenness': betweenness[node_id],
        'Closeness': closeness[node_id],
        'Eigenvector': eigenvector[node_id]
    })

df = pd.DataFrame(data).sort_values(by='Degree', ascending=False)
print("Top 5 Influential Nodes:")
print(df.head())

## 3. Visualization

We visualize the network where:
*   **Node Size** is proportional to Degree Centrality.
*   **Node Color** is proportional to Betweenness Centrality (Darker = Higher Control).

In [ ]:
plt.figure(figsize=(12, 10))

pos = nx.spring_layout(G, seed=42, k=0.15)

# Node Size scaling
node_sizes = [v * 3000 for v in degree.values()]

# Node Color (Betweenness)
node_colors = [betweenness[n] for n in G.nodes()]

nodes = nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=node_colors, 
                               cmap='coolwarm', alpha=0.9)
nx.draw_networkx_edges(G, pos, alpha=0.3)
nx.draw_networkx_labels(G, pos, labels=node_names, font_size=8)

plt.colorbar(nodes, label='Betweenness Centrality')
plt.title("Social Network Analysis: Size=Degree, Color=Betweenness")
plt.axis('off')
plt.show()